In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install unsloth

In [ ]:
!pip install unsloth transformers accelerate torch

## Inference

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load your saved LoRA model (replace "lora_model" with your actual path)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/WR/Qwen_lora_model",  # Your saved directory
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)

# Optional: Merge LoRA adapters if you want a standalone model
# model.save_pretrained_merged("merged_model", tokenizer, save_method="merged_16bit")

In [ ]:
def generate_summary(text, max_length_ratio=0.3, max_new_tokens=1024):
    """
    Generate summary with dynamic length based on input.
    For very long inputs, increase max_new_tokens up to 1024.
    """
    # Calculate target length
    input_token_count = len(tokenizer(text)["input_ids"])
    target_max_tokens = int(input_token_count * max_length_ratio)
    target_max_tokens = min(target_max_tokens, max_new_tokens)  # Cap at 1024

    # Use prompt that encourages full summarization
    prompt = f"""<|im_start|>user\nសូមសង្ខេបអត្ថបទខាងក្រោមជាអត្ថបទខ្លី និងច្បាស់លាស់។ ប្រសិនបើអត្ថបទវាមានភាពវិស្សមរបស់វាទៅ សូមសង្ខេបជាមួយនឹងប្រមាណ {target_max_tokens} ពាក្យ។\n{text}<|im_end|>\n<|im_start|>assistant\n"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.7,
        top_p=0.9,
        top_k=20,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract assistant response
    if "<|im_start|>assistant" in generated_text:
        assistant_response = generated_text.split("<|im_start|>assistant")[1].split("<|im_end|>")[0].strip()
    else:
        assistant_response = generated_text.strip()

    return assistant_response

In [ ]:
long_text = """ប្រធានាធិបតីអាមេរិក លោក ដូណាល់ ត្រាំ បានគិតគូរជាយូរណាស់មកហើយចង់ឱ្យសហរដ្ឋអាមេរិកគ្រប់គ្រងដែនដី Greenland ជាគំនិតមួយដែលធ្វើឱ្យបណ្ដាមេដឹកនាំអឺរ៉ុប និងប្រជាជន Greenland ខឹងសម្បារផង និងសើចចំអកផង។ ប៉ុន្តែនៅបន្ទាប់ពីសហរដ្ឋអាមេរិកវាយប្រហារលើវ៉េណេស៊ុយអេឡា និងចាប់ខ្លួនលោក នីកូឡាស់ ម៉ាឌូរ៉ូ កាលពីថ្ងៃទី៣ ខែមករា វាបានបង្ហាញថាមហិច្ឆតារបស់លោក ត្រាំ ចង់គ្រប់គ្រងដែនដីក្នុងតំបន់អាក់ទិកមួយនេះហាក់មានភាពប្រាកដប្រជាជាងពេលណាៗទាំងអស់។
Greenland ដែលជាតំបន់ស្វយ័តមួយរបស់ដាណឺម៉ាក និងមានប្រជាជនរស់នៅ ៥៧០០០នាក់ បានទទួលឱ្យសហរដ្ឋអាមេរិកដាក់មូលដ្ឋានទ័ពមួយរួចទៅហើយ។ តែរដ្ឋបាលលោក ត្រាំចង់បាន សិទិ្ធអំណាចគ្រប់គ្រងបន្ថែមលើដែនដីមួយនេះ ដោយសារតែ Greenland មានទីភូមិសាស្ត្រយុទ្ធសាស្ត្រយ៉ាងសំខាន់នៅក្នុងតំបន់អាក់ទិកដើម្បីការពារផលប្រយោជន៍សន្តិសុខសហរដ្ឋអាមេរិក និង NATO។ «យើងត្រូវការ Greenland សម្រាប់ការការពារ»។ នេះជាអ្វីដែលលោក ត្រាំ បានប្រាប់កាសែតអាមេរិក The Atlantic កាលពីថ្ងៃទី៤ ខែមករា ជាមួយការអះអាងថា Greenland កំពុងត្រូវបានហ៊ុមព័ទ្ធដោយកប៉ាល់ចិន និងរុស្ស៉ី។ Greenland ក៏សម្បូរផងដែរធនធានធម្មជាតិ ក្នុងនោះរួមមានឧស្ម័នធម្មជាតិ ប្រេង និងរ៉ែដែលត្រូវបានប្រើក្នុងបច្ចេកវិទ្យា និងយោធាដែលកំពុង ក្លាយជាចំណុចក្ដៅមួយក្នុងសង្រ្គាមពាណិជ្ជកម្មរវាងសហរដ្ឋអាមេរិក និងចិន។
តែទោះជាបែបនេះក្ដី រដ្ឋបាលលោក ត្រាំ កំពុងប្រឈមនឹងការប្រឆាំងជំទាស់យ៉ាងខ្លាំងពីបណ្ដាមេដឹកនាំ NATO ក្នុងនោះមានទាំងដាណឺម៉ាដែលនៅតែមានសិទ្ធិអំណាចចាត់ចែងគោលនយោបាយ ការពារជាតិ និងការបរទេសរបស់ Greenland។ នៅក្នុងសេចក្ដីថ្លែងការណ៍មួយកាលពីថ្ងៃទី៦ ខែមករា មេដឹកនាំដាណឺម៉ាក បារាំង អាល្លឺម៉ង់ អ៉ីតាលី ប៉ូឡូញ និងចក្រភពអង់គ្លេសបាននិយាយថា មានតែប្រជាជនដាណឺម៉ាក និងប្រជាជន Greenland តែប៉ុណ្ណោះដែលអាចសម្រេចចិត្តលើបញ្ហាពាក់ព័ន្ធនឹងដាណឺម៉ាក និង Greenland។ ដូច្នេះតើលោក ត្រាំ អាចមានវិធីសាស្ត្រអ្វីខ្លះដើម្បី គ្រប់គ្រង Greenland?
"""

summary = generate_summary(long_text, max_length_ratio=0.3, max_new_tokens=1024)
print("Summary:\n", summary)

Summary:
 user
សូមសង្ខេបអត្ថបទខាងក្រោមជាអត្ថបទខ្លី និងច្បាស់លាស់។ ប្រសិនបើអត្ថបទវាមានភាពវិស្សមរបស់វាទៅ សូមសង្ខេបជាមួយនឹងប្រមាណ 551 ពាក្យ។
ប្រធានាធិបតីអាមេរិក លោក ដូណាល់ ត្រាំ បានគិតគូរជាយូរណាស់មកហើយចង់ឱ្យសហរដ្ឋអាមេរិកគ្រប់គ្រងដែនដី Greenland ជាគំនិតមួយដែលធ្វើឱ្យបណ្ដាមេដឹកនាំអឺរ៉ុប និងប្រជាជន Greenland ខឹងសម្បារផង និងសើចចំអកផង។ ប៉ុន្តែនៅបន្ទាប់ពីសហរដ្ឋអាមេរិកវាយប្រហារលើវ៉េណេស៊ុយអេឡា និងចាប់ខ្លួនលោក នីកូឡាស់ ម៉ាឌូរ៉ូ កាលពីថ្ងៃទី៣ ខែមករា វាបានបង្ហាញថាមហិច្ឆតារបស់លោក ត្រាំ ចង់គ្រប់គ្រងដែនដីក្នុងតំបន់អាក់ទិកមួយនេះហាក់មានភាពប្រាកដប្រជាជាងពេលណាៗទាំងអស់។
Greenland ដែលជាតំបន់ស្វយ័តមួយរបស់ដាណឺម៉ាក និងមានប្រជាជនរស់នៅ ៥៧០០០នាក់ បានទទួលឱ្យសហរដ្ឋអាមេរិកដាក់មូលដ្ឋានទ័ពមួយរួចទៅហើយ។ តែរដ្ឋបាលលោក ត្រាំចង់បាន សិទិ្ធអំណាចគ្រប់គ្រងបន្ថែមលើដែនដីមួយនេះ ដោយសារតែ Greenland មានទីភូមិសាស្ត្រយុទ្ធសាស្ត្រយ៉ាងសំខាន់នៅក្នុងតំបន់អាក់ទិកដើម្បីការពារផលប្រយោជន៍សន្តិសុខសហរដ្ឋអាមេរិក និង NATO។ «យើងត្រូវការ Greenland សម្រាប់ការការពារ»។ នេះជាអ្វីដែលលោក ត្រាំ បានប្រាប់កាសែតអាមេរិក The Atlantic កាលពីថ្ងៃទី៤ ខែមករា ជាមួយការអះអ

In [ ]:
!python --version

Python 3.12.12
